# Exercise 3: Add Decorators for Logging and Timing (Decorator Pattern)

### Objective:
- Learn the **Decorator Pattern** to add reusable cross-cutting concerns (logging, timing).
- Understand how decorators support the **Open/Closed Principle** (open for extension, closed for modification).

### Concept:
In production Data Science pipelines, we want to know:
1. When data loading and cleaning functions start and finish (`@log_action`)
2. How long each operation takes to execute (`@log_time`)

Instead of copying `print()` and `time.time()` statements into every single function, we use Python **decorators** to wrap existing methods without touching their internal code.

## 1. Creating the Decorators

We define:
- `@log_action`: Prints a log message when the decorated method starts and when it finishes.
- `@log_time`: Computes the execution duration in seconds and prints it.

We use `functools.wraps` to preserve the original function's name and docstring.

In [1]:
import time
import functools
import pandas as pd
from abc import ABC, abstractmethod

def log_action(func):
    """Decorator that logs when a method starts and finishes."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # Identify the class name if called as an instance method
        owner_name = args[0].__class__.__name__ if args else ""
        name = f"{owner_name}.{func.__name__}" if owner_name else func.__name__
        
        print(f"[ACTION LOG] ---> STARTING: {name}()")
        result = func(*args, **kwargs)
        print(f"[ACTION LOG] <--- FINISHED: {name}() successfully!")
        return result
    return wrapper


def log_time(func):
    """Decorator that calculates and prints method execution time."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        owner_name = args[0].__class__.__name__ if args else ""
        name = f"{owner_name}.{func.__name__}" if owner_name else func.__name__
        
        start_time = time.time()
        result = func(*args, **kwargs)
        elapsed_time = time.time() - start_time
        
        print(f"[TIME LOG]    TIMER: {name}() took {elapsed_time:.6f} seconds.")
        return result
    return wrapper

## 2. Applying Decorators to CSVReader and DataCleaner

We now enhance our previous classes by decorating their main processing methods.
Notice that the core business logic remains identical — decorators cleanly extend their functionality!

In [2]:
class CSVReader:
    """Enhanced CSVReader equipped with logging and timing decorators."""
    def __init__(self, file_path: str):
        self.file_path = file_path
        self.data = None
        
    @log_action
    @log_time
    def read(self):
        """Loads the CSV file into a pandas DataFrame."""
        self.data = pd.read_csv(self.file_path)
        return self.data
    
    def preview(self, n: int = 5):
        if self.data is not None:
            return self.data.head(n)
        print("No data loaded.")
        return None

In [3]:
# Re-introducing MissingValueStrategy classes for cleaning
class MissingValueStrategy(ABC):
    @abstractmethod
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        pass

class FillMean(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        cleaned_df = df.copy()
        numeric_cols = cleaned_df.select_dtypes(include=['number']).columns
        for col in numeric_cols:
            if cleaned_df[col].isnull().sum() > 0:
                cleaned_df[col] = cleaned_df[col].fillna(cleaned_df[col].mean())
        return cleaned_df

class DataCleaner:
    """Enhanced DataCleaner equipped with logging and timing decorators."""
    def __init__(self, strategy: MissingValueStrategy):
        self.strategy = strategy
        
    def set_strategy(self, strategy: MissingValueStrategy):
        self.strategy = strategy
        
    @log_action
    @log_time
    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        """Executes strategy with automatic logging and timing."""
        return self.strategy.handle(df)

## 3. Testing the Decorated Methods

In [4]:
# Test decorated CSVReader.read()
reader = CSVReader("sample_data.csv")
df = reader.read()
print("\nPreview loaded data:")
display(reader.preview(3))

[ACTION LOG] ---> STARTING: CSVReader.read()
[TIME LOG]    TIMER: CSVReader.read() took 0.004203 seconds.
[ACTION LOG] <--- FINISHED: CSVReader.read() successfully!

Preview loaded data:


,id,name,age,salary,city
0,1,Alice,25.0,50000.0,New York
1,2,Bob,30.0,NaN,Paris
2,3,Charlie,NaN,70000.0,London


In [5]:
# Test decorated DataCleaner.clean()
cleaner = DataCleaner(strategy=FillMean())
cleaned_df = cleaner.clean(df)
print("\nPreview cleaned data:")
display(cleaned_df.head(3))

[ACTION LOG] ---> STARTING: DataCleaner.clean()
[TIME LOG]    TIMER: DataCleaner.clean() took 0.001108 seconds.
[ACTION LOG] <--- FINISHED: DataCleaner.clean() successfully!

Preview cleaned data:


,id,name,age,salary,city
0,1,Alice,25.0,50000.000000,New York
1,2,Bob,30.0,62142.857143,Paris
2,3,Charlie,30.5,70000.000000,London


### Summary:
- Implemented `@log_action` and `@log_time` using Python function closures and `functools.wraps`.
- Successfully stacked multiple decorators on `read()` and `clean()`.
- Demonstrated the **Decorator Pattern** by injecting non-functional requirements (logging, profiling) without altering core logic.